<a href="https://colab.research.google.com/github/brindatenn/phd-prep-summer-2026/blob/main/week1/neural_network_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Neural Network Classification (PyTorch)

**Classification = predicting a class/category** (vs regression, which predicts a number).
- **Binary** - one of two options (spam / not spam).
- **Multi-class** - one of many options (cat / dog / chicken).
- **Multi-label** - can be assigned several labels at once (article tags).

The whole chapter reuses one workflow: **data → build model → pick loss+optimizer → train loop → evaluate → improve**.

## 0. Architecture cheat sheet

The single most useful reference from this chapter - the typical ingredients of a classification network:

| Component | Binary | Multi-class |
|-----------|--------|-------------|
| Input layer shape (`in_features`) | = number of features | same |
| Hidden layers | ≥ 1 | same |
| Neurons per hidden layer | ~10–512 | same |
| **Output shape (`out_features`)** | **1** | **1 per class** |
| Hidden activation | usually **ReLU** | same |
| **Output activation** | **Sigmoid** | **Softmax** |
| **Loss function** | **`nn.BCEWithLogitsLoss`** | **`nn.CrossEntropyLoss`** |
| Optimizer | SGD, Adam, … | same |

Keep this handy — most "which function do I use?" questions are answered here.

## 1. Make data & check shapes

Using `make_circles` - two interleaved rings, a classic non-linear toy problem.

> **Shape errors are the #1 deep learning bug.** Always ask: *"what shape are my inputs, what shape are my outputs?"*

In [17]:
from sklearn.datasets import make_circles

n_samples = 1000
X, y = make_circles(n_samples, noise=0.03, random_state=42)   # X: (1000, 2) features, y: (1000,) labels 0/1

print("X shape:", X.shape, "| y shape:", y.shape)
print("One sample -> X:", X[0], "y:", y[0])   # 2 features in, 1 label out

X shape: (1000, 2) | y shape: (1000,)
One sample -> X: [0.75424625 0.23148074] y: 1


### Turn into tensors + train/test split
PyTorch wants tensors, not NumPy arrays. Split 80/20 so we train on one set and evaluate on unseen data.

In [18]:
import torch
from sklearn.model_selection import train_test_split

# NumPy -> float tensors
X = torch.from_numpy(X).type(torch.float)
y = torch.from_numpy(y).type(torch.float)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(len(X_train), "train |", len(X_test), "test")

800 train | 200 test


## 2. Build a model

Two standard ways to define a model. **Device-agnostic code** first so it runs on GPU if available.

In [19]:
from torch import nn

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

Using: cpu


**Option A — subclass `nn.Module`** (most flexible; almost all PyTorch models do this).
The rule that prevents shape errors: **each layer's `in_features` must equal the previous layer's `out_features`.**

In [20]:
class CircleModelV0(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(in_features=2, out_features=5)  # 2 features in -> 5 hidden units
        self.layer_2 = nn.Linear(in_features=5, out_features=1)  # 5 in -> 1 out (matches y)

    def forward(self, x):
        return self.layer_2(self.layer_1(x))   # data flows layer_1 -> layer_2

model_0 = CircleModelV0().to(device)
model_0

CircleModelV0(
  (layer_1): Linear(in_features=2, out_features=5, bias=True)
  (layer_2): Linear(in_features=5, out_features=1, bias=True)
)

**Option B — `nn.Sequential`** (shorter, but only runs straight through in order — use a subclass when you need custom logic).

```python
model_0 = nn.Sequential(
    nn.Linear(in_features=2, out_features=5),
    nn.Linear(in_features=5, out_features=1),
).to(device)
```

## 2.1 Loss, optimizer & accuracy

- **Loss** measures how *wrong* the model is (lower = better).
- **`nn.BCEWithLogitsLoss`** is preferred for binary: it has the **sigmoid built in** and is more numerically stable
  than `nn.Sigmoid` + `nn.BCELoss`. → **feed it raw logits, not sigmoid outputs.**
- **Accuracy** is a friendlier "how right" metric we track alongside loss.

In [21]:
loss_fn = nn.BCEWithLogitsLoss()                              # sigmoid built in -> takes raw logits
optimizer = torch.optim.SGD(params=model_0.parameters(), lr=0.1)

# Accuracy metric: % of predictions that match the labels
def accuracy_fn(y_true, y_pred):
    correct = torch.eq(y_true, y_pred).sum().item()          # count matches
    return (correct / len(y_pred)) * 100

## 3. logits → prediction probabilities → prediction labels

This conversion is the concept the whole chapter hammers. A model's **raw outputs are called logits**
($y = x \cdot W^{T} + b$). They're hard to read, so:

**Binary:** logits → **`sigmoid`** → probabilities (0–1) → **`round`** → labels (0 or 1).
Rule: prob ≥ 0.5 → class 1, else class 0.

In [22]:
model_0.eval()
with torch.inference_mode():
    y_logits = model_0(X_test.to(device))[:5]          # raw logits

y_pred_probs = torch.sigmoid(y_logits)                 # -> probabilities
y_pred_labels = torch.round(y_pred_probs)              # -> 0/1 labels
print("logits:\n", y_logits.squeeze())
print("probs: ", y_pred_probs.squeeze())
print("labels:", y_pred_labels.squeeze())

logits:
 tensor([-0.1269, -0.0967, -0.1908, -0.1089, -0.1667])
probs:  tensor([0.4683, 0.4758, 0.4524, 0.4728, 0.4584])
labels: tensor([0., 0., 0., 0., 0.])


## 3.2 The training loop (memorise these 5 steps)

Every PyTorch training loop is the same core dance:

1. **Forward pass** - `model(X_train)`
2. **Calculate loss** - `loss_fn(preds, y_train)`
3. **Zero gradients** - `optimizer.zero_grad()` (grads accumulate by default)
4. **Backpropagation** - `loss.backward()`
5. **Step the optimizer** - `optimizer.step()` (gradient descent)

Testing uses `model.eval()` + `torch.inference_mode()` (no gradient tracking = faster).
> Note: this first model will only reach ~50% (random) accuracy (that's the point, it comes next.)

In [23]:
torch.manual_seed(42)
epochs = 100

# Put data on the target device
X_train, y_train = X_train.to(device), y_train.to(device)
X_test,  y_test  = X_test.to(device),  y_test.to(device)

for epoch in range(epochs):
    ### Training
    model_0.train()
    y_logits = model_0(X_train).squeeze()                     # squeeze removes the extra `1` dim
    y_pred = torch.round(torch.sigmoid(y_logits))             # logits -> probs -> labels

    loss = loss_fn(y_logits, y_train)                         # BCEWithLogitsLoss takes raw logits
    acc = accuracy_fn(y_true=y_train, y_pred=y_pred)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    ### Testing
    model_0.eval()
    with torch.inference_mode():
        test_logits = model_0(X_test).squeeze()
        test_pred = torch.round(torch.sigmoid(test_logits))
        test_loss = loss_fn(test_logits, y_test)
        test_acc = accuracy_fn(y_true=y_test, y_pred=test_pred)

    if epoch % 10 == 0:
        print(f"Epoch {epoch} | Loss {loss:.4f} Acc {acc:.1f}% | Test loss {test_loss:.4f} Test acc {test_acc:.1f}%")

Epoch 0 | Loss 0.6957 Acc 50.0% | Test loss 0.6972 Test acc 50.0%
Epoch 10 | Loss 0.6940 Acc 50.0% | Test loss 0.6962 Test acc 50.0%
Epoch 20 | Loss 0.6934 Acc 46.0% | Test loss 0.6959 Test acc 48.5%
Epoch 30 | Loss 0.6932 Acc 49.0% | Test loss 0.6958 Test acc 47.5%
Epoch 40 | Loss 0.6931 Acc 49.5% | Test loss 0.6957 Test acc 46.5%
Epoch 50 | Loss 0.6931 Acc 50.4% | Test loss 0.6957 Test acc 46.5%
Epoch 60 | Loss 0.6931 Acc 50.5% | Test loss 0.6956 Test acc 46.5%
Epoch 70 | Loss 0.6930 Acc 50.5% | Test loss 0.6956 Test acc 46.5%
Epoch 80 | Loss 0.6930 Acc 50.7% | Test loss 0.6955 Test acc 46.5%
Epoch 90 | Loss 0.6930 Acc 50.4% | Test loss 0.6955 Test acc 46.5%


## 4-5. Why it fails, and how to improve a model

The model above sits at ~50% because **linear layers can only draw straight lines** - and you can't split two
circles with a straight line. This is **underfitting** (not learning the patterns).

**Ways to improve a model (all hyperparameters - "experiment, experiment, experiment"):**
- Add more **layers** (deeper) or more **hidden units** (wider)
- Train for **more epochs**
- Change the **learning rate** (too high → overshoots, too low → learns too slowly)
- **Add non-linear activation functions** ← the actual fix here
- Change the loss function; or use **transfer learning** (notebook 06)

> Troubleshooting tip: start small, get a tiny model to *overfit* a small dataset first (proving it *can* learn),
> then scale up. Adding layers/epochs alone won't help here - the missing piece is non-linearity.

## 6. The missing piece: non-linearity (ReLU)

Real data is rarely straight. **Non-linear activations** (like **ReLU**) let a network bend its decision boundary.
Rule of thumb: put them **between hidden layers**. Same data, same everything else - just add `nn.ReLU()`.

In [24]:
class CircleModelV2(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(in_features=2,  out_features=10)
        self.layer_2 = nn.Linear(in_features=10, out_features=10)
        self.layer_3 = nn.Linear(in_features=10, out_features=1)
        self.relu = nn.ReLU()                                  # <- non-linear activation

    def forward(self, x):
        # ReLU interspersed between the linear layers
        return self.layer_3(self.relu(self.layer_2(self.relu(self.layer_1(x)))))

model_3 = CircleModelV2().to(device)

loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model_3.parameters(), lr=0.1)
model_3

CircleModelV2(
  (layer_1): Linear(in_features=2, out_features=10, bias=True)
  (layer_2): Linear(in_features=10, out_features=10, bias=True)
  (layer_3): Linear(in_features=10, out_features=1, bias=True)
  (relu): ReLU()
)

Training `model_3` (same loop as above, ~1000 epochs) now climbs well past 50% - the ReLU lets it curve
around the circles. The training loop code is identical; only the model changed.

## 7. What the activations actually do

Two of the most common activations, built by hand on a toy tensor so you can *see* them:
- **ReLU** - turns negatives to 0, leaves positives unchanged.
- **Sigmoid** - squashes any number into the range (0, 1): $S(x) = \frac{1}{1+e^{-x}}$.

In [25]:
A = torch.arange(-10, 10, 1, dtype=torch.float32)

def relu(x):
    return torch.maximum(torch.tensor(0.), x)          # negatives -> 0

def sigmoid(x):
    return 1 / (1 + torch.exp(-x))                      # squash to (0, 1)

print("input: ", A)
print("relu:  ", relu(A))
print("sigmoid:", sigmoid(A).round(decimals=3))

input:  tensor([-10.,  -9.,  -8.,  -7.,  -6.,  -5.,  -4.,  -3.,  -2.,  -1.,   0.,   1.,
          2.,   3.,   4.,   5.,   6.,   7.,   8.,   9.])
relu:   tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 2., 3., 4., 5., 6., 7.,
        8., 9.])
sigmoid: tensor([0.0000, 0.0000, 0.0000, 0.0010, 0.0020, 0.0070, 0.0180, 0.0470, 0.1190,
        0.2690, 0.5000, 0.7310, 0.8810, 0.9530, 0.9820, 0.9930, 0.9980, 0.9990,
        1.0000, 1.0000])


## 8. Multi-class classification

Now the payoff: more than two classes. Data via `make_blobs` (4 clusters here). Key differences from binary:

- **Labels must be `LongTensor`** (integer class indices) for `CrossEntropyLoss`.
- **Output layer has one neuron per class** (`out_features = NUM_CLASSES`).
- logits → **`softmax(dim=1)`** → probabilities (each row sums to 1) → **`argmax(dim=1)`** → predicted class.

In [26]:
from sklearn.datasets import make_blobs

NUM_CLASSES = 4
NUM_FEATURES = 2

X_blob, y_blob = make_blobs(n_samples=1000, n_features=NUM_FEATURES,
                            centers=NUM_CLASSES, cluster_std=1.5, random_state=42)

X_blob = torch.from_numpy(X_blob).type(torch.float)
y_blob = torch.from_numpy(y_blob).type(torch.LongTensor)      # <- Long/integer labels for CrossEntropyLoss

X_blob_train, X_blob_test, y_blob_train, y_blob_test = train_test_split(
    X_blob, y_blob, test_size=0.2, random_state=42)

### Build the multi-class model
A reusable pattern: pass shapes in as arguments so the same class works for any dataset.

In [27]:
class BlobModel(nn.Module):
    def __init__(self, input_features, output_features, hidden_units=8):
        super().__init__()
        self.linear_layer_stack = nn.Sequential(
            nn.Linear(in_features=input_features, out_features=hidden_units),
            # nn.ReLU(),   # <- add if the data needs non-linearity (blobs are linearly separable, so optional)
            nn.Linear(in_features=hidden_units, out_features=hidden_units),
            # nn.ReLU(),
            nn.Linear(in_features=hidden_units, out_features=output_features),  # one output per class
        )
    def forward(self, x):
        return self.linear_layer_stack(x)

model_4 = BlobModel(input_features=NUM_FEATURES, output_features=NUM_CLASSES, hidden_units=8).to(device)

loss_fn = nn.CrossEntropyLoss()                              # multi-class loss (takes raw logits + Long labels)
optimizer = torch.optim.SGD(model_4.parameters(), lr=0.1)
model_4

BlobModel(
  (linear_layer_stack): Sequential(
    (0): Linear(in_features=2, out_features=8, bias=True)
    (1): Linear(in_features=8, out_features=8, bias=True)
    (2): Linear(in_features=8, out_features=4, bias=True)
  )
)

### logits → probabilities → labels (multi-class)
`softmax` across `dim=1` makes each sample's class scores sum to 1; `argmax` picks the winning class.

In [28]:
model_4.eval()
with torch.inference_mode():
    y_logits = model_4(X_blob_test.to(device))

y_pred_probs = torch.softmax(y_logits, dim=1)                # each row sums to 1
y_preds = y_pred_probs.argmax(dim=1)                         # index of highest prob = predicted class
print("probs[0]:", y_pred_probs[0], "-> sums to", y_pred_probs[0].sum().item())
print("pred class[0]:", y_preds[0].item())
# Shortcut: torch.argmax(y_logits, dim=1) skips softmax (no probabilities kept, but same label)

probs[0]: tensor([0.0644, 0.3126, 0.1307, 0.4923]) -> sums to 1.0
pred class[0]: 3


### Multi-class training loop
Same 5 steps — only the logits→labels conversion (softmax/argmax) and the loss (CrossEntropyLoss) differ.

In [29]:
torch.manual_seed(42)
epochs = 100

X_blob_train, y_blob_train = X_blob_train.to(device), y_blob_train.to(device)
X_blob_test,  y_blob_test  = X_blob_test.to(device),  y_blob_test.to(device)

for epoch in range(epochs):
    ### Training
    model_4.train()
    y_logits = model_4(X_blob_train)
    y_pred = torch.softmax(y_logits, dim=1).argmax(dim=1)     # logits -> probs -> labels

    loss = loss_fn(y_logits, y_blob_train)                   # CrossEntropyLoss takes raw logits
    acc = accuracy_fn(y_true=y_blob_train, y_pred=y_pred)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    ### Testing
    model_4.eval()
    with torch.inference_mode():
        test_logits = model_4(X_blob_test)
        test_pred = torch.softmax(test_logits, dim=1).argmax(dim=1)
        test_loss = loss_fn(test_logits, y_blob_test)
        test_acc = accuracy_fn(y_true=y_blob_test, y_pred=test_pred)

    if epoch % 10 == 0:
        print(f"Epoch {epoch} | Loss {loss:.4f} Acc {acc:.1f}% | Test loss {test_loss:.4f} Test acc {test_acc:.1f}%")

Epoch 0 | Loss 2.1384 Acc 1.8% | Test loss 1.0902 Test acc 48.0%
Epoch 10 | Loss 0.3744 Acc 89.0% | Test loss 0.3299 Test acc 80.5%
Epoch 20 | Loss 0.1264 Acc 98.8% | Test loss 0.1147 Test acc 99.0%
Epoch 30 | Loss 0.0772 Acc 99.0% | Test loss 0.0708 Test acc 99.5%
Epoch 40 | Loss 0.0588 Acc 99.0% | Test loss 0.0521 Test acc 99.5%
Epoch 50 | Loss 0.0494 Acc 99.0% | Test loss 0.0422 Test acc 99.5%
Epoch 60 | Loss 0.0437 Acc 99.0% | Test loss 0.0362 Test acc 99.5%
Epoch 70 | Loss 0.0400 Acc 99.0% | Test loss 0.0321 Test acc 99.5%
Epoch 80 | Loss 0.0374 Acc 99.0% | Test loss 0.0292 Test acc 99.5%
Epoch 90 | Loss 0.0354 Acc 99.0% | Test loss 0.0271 Test acc 99.5%


## 9. Evaluation metrics beyond accuracy

Accuracy alone can mislead (especially on **imbalanced** data). Common classification metrics:

| Metric | What it tells you |
|--------|-------------------|
| **Accuracy** | overall % correct — good default, poor on imbalanced classes |
| **Precision** | of predicted-positives, how many were right (↑ = fewer false positives) |
| **Recall** | of actual-positives, how many we caught (↑ = fewer false negatives) |
| **F1-score** | balance of precision & recall (1 best, 0 worst) |
| **Confusion matrix** | table of predicted vs true — diagonal = correct |

`torchmetrics` and `sklearn.metrics` both implement these.

In [30]:
# Example with torchmetrics
!pip install torchmetrics
from torchmetrics import Accuracy

acc_metric = Accuracy(task="multiclass", num_classes=4).to(device)
print("\n")
print("torchmetrics accuracy:", acc_metric(y_preds, y_blob_test).item())



torchmetrics accuracy: 0.014999999664723873


## Quick recap

- Workflow is always: **data → model → loss+optimizer → train loop → evaluate → improve**.
- Match shapes: each layer's `in_features` = previous `out_features`; use `.squeeze()` to fix stray `1` dims.
- **Binary:** `BCEWithLogitsLoss` (feed raw logits) → `sigmoid` → `round`.
- **Multi-class:** `CrossEntropyLoss` (raw logits + **Long** labels), one output per class → `softmax(dim=1)` → `argmax(dim=1)`.
- Training loop = **forward → loss → zero_grad → backward → step**; eval under `inference_mode()`.
- Linear layers draw straight lines only — add **non-linear activations (ReLU)** between hidden layers to fit curved data.
- Look past accuracy (precision / recall / F1 / confusion matrix), especially on imbalanced data.